# Data Cleaning
## Overview
This notebook cleans the raw USGS annual peak-flow dataset for flood frequency analysis. The raw RDB file is imported, and the USGS metadata and data-type rows are handled appropriately. Relevant variables are selected and renamed, dates and discharge values are converted into suitable data types, and peak discharge is converted from cubic feet per second to cubic metres per second. Records with missing year or discharge values are removed, and the study period is restricted to 1950–2020. The dataset is then checked for missing values, duplicate years, record length, and possible outliers before the final clean dataset is saved for statistical and probability-distribution analysis.


In [ ]:
# Import Required Libraries
from pathlib import Path
import pandas as pd

## 1. Define input and output file paths

In [ ]:
# Change this station number for a different study area
SITE_NO = "06214500"

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_file = RAW_DIR / f"usgs_{SITE_NO}_annual_peak_flow.rdb"
output_file = PROCESSED_DIR / f"usgs_{SITE_NO}_clean_flood_data.csv"

if not raw_file.exists():
    raise FileNotFoundError(
        f"{raw_file} was not found. "
        "Run 01_download_annual_peak_flow.ipynb first."
    )

print("Input file:", raw_file)
print("Output file:", output_file)

## 2. Read the raw USGS RDB file

In [ ]:
df = pd.read_csv(
    raw_file,
    sep="\t",
    comment="#",
    dtype=str
)

df.head()

## 3. Remove the USGS data-type row

In [ ]:
# The first data row contains values such as 5s, 15s, 10d, and 8n
df = df[df["agency_cd"] != "5s"].copy()

df.head()

## 4. Keep useful columns

In [ ]:
required_columns = [
    "site_no",
    "peak_dt",
    "peak_va",
    "gage_ht",
    "peak_cd",
]

available_columns = [
    column for column in required_columns
    if column in df.columns
]

df = df[available_columns].copy()

print("Selected columns:")
print(df.columns.tolist())

## 5. Rename columns

In [ ]:
df = df.rename(
    columns={
        "site_no": "Station_ID",
        "peak_dt": "Date",
        "peak_va": "Peak_Flow_cfs",
        "gage_ht": "Gage_Height_ft",
        "peak_cd": "Peak_Code",
    }
)

df.head()

## 6. Convert data types

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Year"] = df["Date"].dt.year

df["Peak_Flow_cfs"] = pd.to_numeric(
    df["Peak_Flow_cfs"],
    errors="coerce"
)

if "Gage_Height_ft" in df.columns:
    df["Gage_Height_ft"] = pd.to_numeric(
        df["Gage_Height_ft"],
        errors="coerce"
    )

## 7. Convert discharge from cfs to m³/s

In [ ]:
df["Peak_Flow_m3s"] = (
    df["Peak_Flow_cfs"] * 0.0283168
)

df.head()

## 8. Remove missing values

In [ ]:
df = df.dropna(
    subset=["Year", "Peak_Flow_m3s"]
).copy()

df["Year"] = df["Year"].astype(int)

print("Missing values:")
print(df[["Year", "Peak_Flow_m3s"]].isnull().sum())

## 9. Select and rename the analysis columns

In [ ]:
df_clean = df[
    ["Year", "Peak_Flow_m3s"]
].copy()

df_clean = df_clean.rename(
    columns={
        "Peak_Flow_m3s": "Peak_Flow"
    }
)

df_clean.head(3)

## 10. Select the study period

In [ ]:
START_YEAR = 1950
END_YEAR = 2020

df_clean = (
    df_clean[
        df_clean["Year"].between(
            START_YEAR,
            END_YEAR
        )
    ]
    .sort_values("Year")
    .reset_index(drop=True)
)

print("First three records:")
print(df_clean.head(3))
print()
print("Last three records:")
print(df_clean.tail(3))

## 11. Check missing values

In [ ]:
print(df_clean.isnull().sum())

## 12. Check duplicate years

In [ ]:
duplicate_years = df_clean[
    df_clean.duplicated(
        subset="Year",
        keep=False
    )
]

if duplicate_years.empty:
    print("No duplicate years found.")
else:
    print("Duplicate years found:")
    display(duplicate_years)

## 13. Check data length and year range

In [ ]:
print(f"Number of records: {len(df_clean)}")
print(f"Start year: {df_clean['Year'].min()}")
print(f"End year: {df_clean['Year'].max()}")

expected_years = END_YEAR - START_YEAR + 1
print(f"Expected number of years: {expected_years}")

## 14. Identify possible outliers using the IQR method

The IQR method is used only as a screening method. Possible flood outliers should not be removed automatically without reviewing station records and hydrological evidence.

In [ ]:
Q1 = df_clean["Peak_Flow"].quantile(0.25)
Q3 = df_clean["Peak_Flow"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

possible_outliers = df_clean[
    (df_clean["Peak_Flow"] < lower_limit)
    | (df_clean["Peak_Flow"] > upper_limit)
]

print(f"Q1: {Q1:.3f} m³/s")
print(f"Q3: {Q3:.3f} m³/s")
print(f"IQR: {IQR:.3f} m³/s")
print(f"Lower limit: {lower_limit:.3f} m³/s")
print(f"Upper limit: {upper_limit:.3f} m³/s")
print()

if possible_outliers.empty:
    print("No possible outliers were identified.")
else:
    print("Possible outliers:")
    display(possible_outliers)

## 15. Preview the final clean dataset

In [ ]:
print(df_clean.head())
print()
print(df_clean.tail())

## 16. Save the final clean dataset

In [ ]:
df_clean.to_csv(
    output_file,
    index=False
)

print("Clean flood dataset saved successfully.")
print(f"File location: {output_file}")